In [1]:
from julia.api import Julia
from julia import Main

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import os

import pathlib
from pathlib import Path
import json

#fitters

import pybobyqa
import time
import cma
import csv

def to_float64(df):
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].astype("float64")
    return df

Which Fit?

In [2]:
fit_name = "Final"

approximate_total_xsec = True

N_replicas = 100
vary_data_points = True
use_pdf_shift = True

output_csv_name = "replica_0410.csv"
use_random_seed = True
replica_seed = 12345


Read Files

In [3]:
TMD_fitting_root = "../"
def include(name):
    path = os.path.join(TMD_fitting_root, name)
    Main.eval(f'include(raw"{path}")')

include(f"Cards/{fit_name}.jl")
include(f"DY/DY_table_{Main.flavor_scheme}.jl")

# Data
file_root = f"../Data/{Main.data_name}/Cutted/DY"
table_root = f"../Tables/{Main.table_name}/DY"
total_root = f"../Data/DY_total_xsec/{Main.pdf_name}"
chi2_root = Path("../Data/Chi2_Matrix")

initial_params = Main.initial_params


By file or by experiment?

In [4]:
data_selections = "by_experiment"  # "by_file" or "by_experiment"

In [5]:
experiments =[
    'ATLAS_7',
    'ATLAS_8', 
    #'ATLAS_13', 
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    #'PHENIX',
    'STAR'
]

In [6]:
from pathlib import Path

file_excludes = [
    "E772/E772-5Q6.csv",
    "E772/E772-6Q7.csv",
    "E772/E772-7Q8.csv",
    "E772/E772-8Q9.csv",
]

if data_selections == "by_experiment":
    file_names = []
    for experiment in experiments:
        exp_dir = Path(file_root) / experiment
        for p in sorted(exp_dir.glob("*.csv")):
            if str(experiment + "/" + p.name) in file_excludes:
                continue
            file_names.append(str(Path(experiment) / p.name)) 

display(file_names)

['ATLAS_7\\ATLAS7-00y10.csv',
 'ATLAS_7\\ATLAS7-10y20.csv',
 'ATLAS_7\\ATLAS7-20y24.csv',
 'ATLAS_8\\ATLAS8-00y04.csv',
 'ATLAS_8\\ATLAS8-04y08.csv',
 'ATLAS_8\\ATLAS8-08y12.csv',
 'ATLAS_8\\ATLAS8-116Q150.csv',
 'ATLAS_8\\ATLAS8-12y16.csv',
 'ATLAS_8\\ATLAS8-16y20.csv',
 'ATLAS_8\\ATLAS8-20y24.csv',
 'ATLAS_8\\ATLAS8-46Q66.csv',
 'CDF_I\\CDF1.csv',
 'CDF_II\\CDF2.csv',
 'CMS_7\\CMS7.csv',
 'CMS_8\\CMS8.csv',
 'CMS_13\\CMS13-00y04.csv',
 'CMS_13\\CMS13-04y08.csv',
 'CMS_13\\CMS13-08y12.csv',
 'CMS_13\\CMS13-106Q170.csv',
 'CMS_13\\CMS13-12y16.csv',
 'CMS_13\\CMS13-16y24.csv',
 'CMS_13\\CMS13-170Q350.csv',
 'CMS_13\\CMS13-350Q1000.csv',
 'D0_I\\D01.csv',
 'D0_II\\D02.csv',
 'D0_II_mu\\D02mu.csv',
 'E288\\E228-200-4Q5.csv',
 'E288\\E228-200-5Q6.csv',
 'E288\\E228-200-6Q7.csv',
 'E288\\E228-200-7Q8.csv',
 'E288\\E228-200-8Q9.csv',
 'E288\\E228-300-11Q12.csv',
 'E288\\E228-300-4Q5.csv',
 'E288\\E228-300-5Q6.csv',
 'E288\\E228-300-6Q7.csv',
 'E288\\E228-300-7Q8.csv',
 'E288\\E228-300-8Q9.cs

Read Data

In [7]:
data_list = dict()
df_total_xsec = to_float64(pd.read_csv(f"{total_root}.csv"))
total_xsec_names = df_total_xsec['name'].tolist()

for file in tqdm(file_names):

    df_data = to_float64(pd.read_csv(f"{file_root}/{file}"))
    data_list[file] = df_data

    name_short = Path(file).stem
    if name_short in total_xsec_names:
        total_xsec = df_total_xsec[df_total_xsec['name'] == name_short]["total_xsec"].values[0]
        data_list[file]['total_xsec'] = total_xsec*np.ones(len(data_list[file]))
        print(f"{name_short}'s total xsec = {total_xsec} added")

normalize_file = lambda p: str(p).replace('\\', '/')
chi2_index_df = pd.read_csv(chi2_root / "Index.csv")
chi2_index_df["file"] = chi2_index_df["file"].astype(str).map(normalize_file)
required_cols = {"global_index", "file", "local_index"}
missing_cols = required_cols - set(chi2_index_df.columns)
if missing_cols:
    raise ValueError(f"Index.csv is missing required columns: {sorted(missing_cols)}")

chi2_index_df = chi2_index_df.sort_values("global_index").reset_index(drop=True)
expected_global = np.arange(len(chi2_index_df), dtype=int)
actual_global = chi2_index_df["global_index"].to_numpy(dtype=int)
if not np.array_equal(actual_global, expected_global):
    raise ValueError("Index.csv global_index must be contiguous and start at 0.")

file_set = {normalize_file(file) for file in file_names}
index_file_set = set(chi2_index_df["file"].unique())
missing_from_index = sorted(file_set - index_file_set)
extra_in_index = sorted(index_file_set - file_set)
if missing_from_index:
    raise ValueError(f"Index.csv is missing active fit files: {missing_from_index}")
if extra_in_index:
    raise ValueError(f"Index.csv contains files not used by this fit: {extra_in_index}")

chi2_file_positions = {}
chi2_file_local_indices = {}
for file, group in chi2_index_df.groupby("file", sort=False):
    chi2_file_positions[file] = group["global_index"].to_numpy(dtype=int)
    chi2_file_local_indices[file] = group["local_index"].to_numpy(dtype=int)

normalized_data_lists = {normalize_file(file): data_list[file] for file in file_names}
for file, local_indices in chi2_file_local_indices.items():
    file_len = len(normalized_data_lists[file])
    if np.any(local_indices < 0) or np.any(local_indices >= file_len):
        raise IndexError(f"Index.csv local_index out of bounds for {file}")

Total_inverse = to_float64(pd.read_csv(chi2_root / "Total_inverse.csv")).to_numpy(dtype=float)
if Total_inverse.shape != (len(chi2_index_df), len(chi2_index_df)):
    raise ValueError(
        f"Total_inverse.csv shape {Total_inverse.shape} does not match Index.csv length {len(chi2_index_df)}"
    )
Total_inverse = 0.5 * (Total_inverse + Total_inverse.T)


 67%|██████▋   | 38/57 [00:00<00:00, 372.55it/s]

ATLAS7-00y10's total xsec = 1.0 added
ATLAS7-10y20's total xsec = 1.0 added
ATLAS7-20y24's total xsec = 1.0 added
CMS7's total xsec = 1.0 added
CMS8's total xsec = 1.0 added
D02's total xsec = 1.0 added
D02mu's total xsec = 1.0 added


100%|██████████| 57/57 [00:00<00:00, 238.49it/s]


Prediction

In [8]:
Params = Main.Params_Struct(*[np.float32(x) for x in initial_params]) 
#Main.set_params(Main.VRAM, Params) 

for i in range(10):
    Params = Main.Params_Struct(*[np.float32(x) for x in initial_params])                  
    predictions,t = Main.xsec_dict(Main.rel_paths, Main.VRAM)
    print(round(t*1000,2), "ms")

436.06 ms
39.04 ms
37.55 ms
37.62 ms
37.47 ms
37.75 ms
38.69 ms
37.54 ms
37.51 ms
47.37 ms


In [9]:
def get_file_length():

    file_lengths = dict()

    for file in file_names:

        df = to_float64(pd.read_csv(f"{file_root}/{file}"))

        file_lengths[file] = len(df)

    return file_lengths

file_lengths = get_file_length()

In [10]:
def _norm(p: str) -> str:
    return os.path.normpath(p).replace('\\', '/')

def prediction_reformat(predictions):
    preds = {_norm(k): v for k, v in predictions.items()}  # normalize keys once
    df_predictions = {}

    for file in file_names:
        n = file_lengths[file]
        base = os.path.splitext(file)[0]
        xs = []
        for i in range(n):
            table_path = _norm(os.path.join(table_root, f"{base}/{i}.jls"))
            xs.append(preds[table_path])
        xs = np.asarray(xs, dtype=float)
        df_predictions[file] = xs.copy()

        if approximate_total_xsec == True and Path(file).stem in total_xsec_names:
            data_xsec = data_list[file]["xsec"].to_numpy(dtype=float)
            qT_bin_size = (
                data_list[file]["qT_max"].to_numpy(dtype=float)
                - data_list[file]["qT_min"].to_numpy(dtype=float)
            )
            weighted_data = np.sum(data_xsec * qT_bin_size)
            weighted_theory = np.sum(xs * qT_bin_size)
            scale = weighted_data / weighted_theory
            df_predictions[file] = scale * xs
            #display(file, 1/scale)

    return df_predictions

df_predictions = prediction_reformat(predictions)


In [11]:
def clone_data_list(source):
    return {file: df.copy(deep=True) for file, df in source.items()}

nominal_data_list = clone_data_list(data_list)

zero_pdf_shift_vector = np.zeros((len(chi2_index_df), 1), dtype=np.float64)

if use_pdf_shift:
    pdf_rep_path = chi2_root / "PDF_rep.csv"
    PDF_rep = to_float64(pd.read_csv(pdf_rep_path)).to_numpy(dtype=float)
    if PDF_rep.ndim == 1:
        PDF_rep = PDF_rep.reshape(-1, 1)
    if PDF_rep.shape[0] != len(chi2_index_df):
        raise ValueError(
            f"PDF_rep.csv shape {PDF_rep.shape} is incompatible with Index.csv length {len(chi2_index_df)}"
        )
    print(f"Loaded Gaussian PDF shift basis {PDF_rep.shape} from {pdf_rep_path}")
else:
    PDF_rep = np.zeros((len(chi2_index_df), 0), dtype=float)
    print("Gaussian PDF shifts are disabled.")

current_pdf_replica_id = None
current_pdf_shift_vector = zero_pdf_shift_vector.copy()
current_pdf_seed = None


Loaded Gaussian PDF shift basis (465, 99) from ..\Data\Chi2_Matrix\PDF_rep.csv


Chi2

In [12]:
def _coerce_index_df(index_source):
    if isinstance(index_source, pd.DataFrame):
        index_df = index_source.copy()
    else:
        index_df = pd.DataFrame(index_source)

    required_cols = ["global_index", "file", "local_index"]
    missing_cols = [col for col in required_cols if col not in index_df.columns]
    if missing_cols:
        raise ValueError(f"Index source is missing required columns: {missing_cols}")

    index_df = index_df.copy()
    index_df["file"] = index_df["file"].astype(str).map(_norm)
    index_df = index_df.sort_values("global_index").reset_index(drop=True)
    return index_df


def build_indexed_column_vector(index_source, arrays_by_file):
    index_df = _coerce_index_df(index_source)
    normalized_arrays = {_norm(file): values for file, values in arrays_by_file.items()}
    vector = np.full((len(index_df), 1), np.nan, dtype=float)

    for file, group in index_df.groupby("file", sort=False):
        if file not in normalized_arrays:
            raise KeyError(f"Missing values for {file} in arrays_by_file")

        values = np.asarray(normalized_arrays[file], dtype=float).reshape(-1)
        local_indices = group["local_index"].to_numpy(dtype=int)
        if np.any(local_indices < 0) or np.any(local_indices >= len(values)):
            raise IndexError(f"Indexed local positions are out of bounds for {file}")

        global_indices = group["global_index"].to_numpy(dtype=int)
        vector[global_indices, 0] = values[local_indices]

    if np.isnan(vector).any():
        raise ValueError("Indexed column vector has unfilled entries. Check Index.csv coverage.")

    return vector


def get_prediction_column_vector(index_source, df_predictions):
    return build_indexed_column_vector(index_source, df_predictions)


def get_data_column_vector(index_source, source_data_list=None):
    if source_data_list is None:
        source_data_list = data_list
    arrays_by_file = {
        file: source_data_list[file]["xsec"].to_numpy(dtype=float)
        for file in source_data_list
    }
    return build_indexed_column_vector(index_source, arrays_by_file)


def apply_pdf_shift_to_prediction_vector(prediction_vector, pdf_shift_vector):
    prediction_vector = np.asarray(prediction_vector, dtype=float).reshape(len(chi2_index_df), 1)
    pdf_shift_vector = np.asarray(pdf_shift_vector, dtype=float).reshape(len(chi2_index_df), 1)
    return prediction_vector + pdf_shift_vector


def get_chi2dN_from_prediction_vector(prediction_vector, source_data_list=None):
    N_list = dict()
    chi2dN_list = dict()

    if source_data_list is None:
        source_data_list = data_list

    prediction_vector = np.asarray(prediction_vector, dtype=float).reshape(len(chi2_index_df), 1)
    current_data_vector = get_data_column_vector(chi2_index_df, source_data_list)
    diff_vector = current_data_vector - prediction_vector
    weighted_diff = Total_inverse @ diff_vector

    chi2_total = float((diff_vector.T @ weighted_diff)[0, 0])
    N_total = int(diff_vector.shape[0])

    point_contributions = diff_vector[:, 0] * weighted_diff[:, 0]

    for file in chi2_index_df["file"].unique():
        positions = chi2_file_positions[file]
        N = len(positions)
        chi2_file = float(np.sum(point_contributions[positions]))
        chi2dN_list[file] = float(round(chi2_file / N, 3))
        N_list[file] = N

    chi2dN = chi2_total / N_total
    return chi2dN, chi2dN_list, N_list


def get_chi2dN(df_predictions, pdf_shift_vector=None, source_data_list=None):
    prediction_vector = get_prediction_column_vector(chi2_index_df, df_predictions)
    if pdf_shift_vector is not None:
        prediction_vector = apply_pdf_shift_to_prediction_vector(prediction_vector, pdf_shift_vector)
    return get_chi2dN_from_prediction_vector(prediction_vector, source_data_list)

chi2dN, chi2dN_list, N_list = get_chi2dN(df_predictions, current_pdf_shift_vector)

print(f"Total chi^2/N = {chi2dN:.3f}")
display(chi2dN_list)


Total chi^2/N = 1.120


{'ATLAS_7/ATLAS7-00y10.csv': 4.378,
 'ATLAS_7/ATLAS7-10y20.csv': 5.795,
 'ATLAS_7/ATLAS7-20y24.csv': 1.304,
 'ATLAS_8/ATLAS8-00y04.csv': 3.79,
 'ATLAS_8/ATLAS8-04y08.csv': 0.226,
 'ATLAS_8/ATLAS8-08y12.csv': 1.222,
 'ATLAS_8/ATLAS8-116Q150.csv': 0.674,
 'ATLAS_8/ATLAS8-12y16.csv': 3.169,
 'ATLAS_8/ATLAS8-16y20.csv': 0.84,
 'ATLAS_8/ATLAS8-20y24.csv': 1.442,
 'ATLAS_8/ATLAS8-46Q66.csv': 2.303,
 'CDF_I/CDF1.csv': 0.637,
 'CDF_II/CDF2.csv': 1.407,
 'CMS_7/CMS7.csv': 1.414,
 'CMS_8/CMS8.csv': 0.753,
 'CMS_13/CMS13-00y04.csv': 1.678,
 'CMS_13/CMS13-04y08.csv': 0.95,
 'CMS_13/CMS13-08y12.csv': 0.464,
 'CMS_13/CMS13-106Q170.csv': 2.993,
 'CMS_13/CMS13-12y16.csv': 0.225,
 'CMS_13/CMS13-16y24.csv': 0.227,
 'CMS_13/CMS13-170Q350.csv': 1.631,
 'CMS_13/CMS13-350Q1000.csv': 1.717,
 'D0_I/D01.csv': 0.538,
 'D0_II/D02.csv': 1.124,
 'D0_II_mu/D02mu.csv': 0.493,
 'E288/E228-200-4Q5.csv': 0.378,
 'E288/E228-200-5Q6.csv': 0.234,
 'E288/E228-200-6Q7.csv': 0.314,
 'E288/E228-200-7Q8.csv': 0.223,
 'E288/E22

Objective

In [13]:
def objective(params):
    try:
        params_cl = Main.Params_Struct(*[np.float32(x) for x in params])
        Main.set_params(Main.VRAM, params_cl)

        predictions, t = Main.xsec_dict(Main.rel_paths, Main.VRAM)

        df_predictions = prediction_reformat(predictions)
        chi2dN, _, _ = get_chi2dN(df_predictions, current_pdf_shift_vector)

        if not np.isfinite(chi2dN):
            return 1e5
        return chi2dN

    except Exception as e:
        return 1e5

print(objective(initial_params))

# freeze parameters

initial_params = np.asarray(initial_params, float)
frozen_idx = np.asarray(Main.frozen_indices, int)

dim_full = len(initial_params)
mask = np.ones(dim_full, dtype=bool)
mask[frozen_idx] = False
free_idx = np.where(mask)[0]   # indices that WILL be fitted

frozen_vals = initial_params[frozen_idx].copy()

def objective_with_freeze(params_free):

    params_free = np.asarray(params_free, float)
    full = initial_params.copy()
    full[free_idx] = params_free
    full[frozen_idx] = frozen_vals
    return objective(full)

print(objective_with_freeze(initial_params[free_idx]))


1.1197770074265536
1.119774896225828


Bounds

In [14]:
bounds_raw = np.asarray(Main.bounds_raw, float)[free_idx]

lower_bounds, upper_bounds = np.array(list(zip(*bounds_raw)))

def objective_normalized(params):

    normalized_params = lower_bounds + params * (upper_bounds - lower_bounds)

    return objective_with_freeze(normalized_params)

def objective_log(params):
    return np.log10(objective_normalized(params))

def normalize_params(params):
    return (params - lower_bounds) / (upper_bounds - lower_bounds)

def denormalize_params(params):
    return lower_bounds + params * (upper_bounds - lower_bounds)

theta0 = normalize_params(np.array(initial_params)[free_idx])
dim = len(bounds_raw)

## Replica Refit Workflow

Each fit replica draws:

1. A pseudo-dataset from the experimental error columns, with independent Gaussian draws for `error_unc_*` and one shared Gaussian per dataset-level `error_add_*` or `error_mult_*` source.
2. A Gaussian PDF shift built as `PDF_rep.csv @ z`, with `z ~ N(0, I)` in the PDF eigenspace.

The sampled PDF shift vector is added to the 465-point prediction vector before evaluating `Ï‡Â²`.


In [15]:
def make_replica_rng(replica_id):
    if use_random_seed:
        return np.random.default_rng()
    return np.random.default_rng(np.random.SeedSequence([replica_seed, replica_id]))

randomize_start = False
alpha = 9.0

local_stage1_maxfun = 220
local_stage2_maxfun = 120
local_stage1_rhobeg = 0.07
local_stage1_rhoend = 5e-5
local_stage2_rhobeg = 0.03
local_stage2_rhoend = 1e-6

rescue_stage1_maxfun = 180
rescue_stage2_maxfun = 120
rescue_stage1_rhobeg = 0.09
rescue_stage1_rhoend = 1e-4
rescue_stage2_rhobeg = 0.04
rescue_stage2_rhoend = 1e-6

base_jitter_sigma = 0.05
previous_jitter_sigma = 0.03

rescue_enabled = True
rescue_chi2dN_threshold = 2.1

results_path = Path("replica_data") / output_csv_name
results_path.parent.mkdir(parents=True, exist_ok=True)

param_columns = [f"param_{i}" for i in range(dim_full)]
pdf_seed_column = "pdf_shift_seed"


In [16]:
def _replica_error_columns(df, prefix):
    return [col for col in df.columns if str(col).strip().startswith(prefix)]


def generate_experimental_replica_data(rng):
    replica_data = clone_data_list(nominal_data_list)
    if not vary_data_points:
        return replica_data

    for file in file_names:
        nominal_df = nominal_data_list[file]
        xsecs = nominal_df["xsec"].to_numpy(dtype=np.float64)
        replica_xsecs = xsecs.copy()

        unc_cols = _replica_error_columns(nominal_df, "error_unc_")
        if unc_cols:
            unc_errors = nominal_df[unc_cols].to_numpy(dtype=np.float64)
            replica_xsecs += np.sum(
                unc_errors * rng.standard_normal(size=unc_errors.shape),
                axis=1,
            )

        add_cols = _replica_error_columns(nominal_df, "error_add_")
        if add_cols:
            add_errors = nominal_df[add_cols].to_numpy(dtype=np.float64)
            replica_xsecs += add_errors @ rng.standard_normal(add_errors.shape[1])

        mult_cols = _replica_error_columns(nominal_df, "error_mult_")
        if mult_cols:
            mult_errors = nominal_df[mult_cols].to_numpy(dtype=np.float64)
            replica_xsecs += xsecs * (mult_errors @ rng.standard_normal(mult_errors.shape[1]))

        replica_data[file]["xsec"] = replica_xsecs

    return replica_data


def sample_pdf_replica(rng, replica_id):
    if (not use_pdf_shift) or PDF_rep.shape[1] == 0:
        return "off", zero_pdf_shift_vector.copy(), None

    pdf_seed = int(rng.integers(0, np.iinfo(np.int64).max, dtype=np.int64))
    pdf_rng = np.random.default_rng(pdf_seed)
    z = pdf_rng.standard_normal(PDF_rep.shape[1])
    pdf_shift_vector = (PDF_rep @ z).reshape(len(chi2_index_df), 1)
    return "gaussian", pdf_shift_vector, pdf_seed


In [17]:
from types import SimpleNamespace

def draw_theta0(rng):
    if not randomize_start:
        return theta0.copy()

    while True:
        cand = rng.beta(alpha, alpha, size=dim)
        val = float(objective_log(cand))
        if val < 1.0:
            return cand

def jitter_theta(theta, sigma, rng):
    return np.clip(theta + rng.normal(scale=sigma, size=dim), 0.0, 1.0)

def dedupe_theta_starts(starts, atol=1e-10):
    unique = []
    for theta in starts:
        if theta is None:
            continue
        if not any(np.allclose(theta, prev, atol=atol, rtol=0.0) for prev in unique):
            unique.append(theta.copy())
    return unique

def build_replica_starts(rng):
    base_theta = draw_theta0(rng)
    starts = [
        base_theta,
        jitter_theta(base_theta, base_jitter_sigma, rng),
        jitter_theta(base_theta, previous_jitter_sigma, rng),
    ]
    return dedupe_theta_starts(starts)[:3]

def fallback_result(theta_start, exc):
    return SimpleNamespace(
        x=np.clip(theta_start.copy(), 0.0, 1.0),
        f=float(objective_log(theta_start)),
        nf=0,
        flag=-999,
        message=str(exc),
    )

def solve_replica_stage(theta_start, *, maxfun, rhobeg, rhoend, seek_global_minimum):
    try:
        return pybobyqa.solve(
            objective_log,
            np.clip(theta_start, 0.0, 1.0),
            bounds=(np.zeros(dim), np.ones(dim)),
            maxfun=maxfun,
            rhobeg=rhobeg,
            rhoend=rhoend,
            scaling_within_bounds=True,
            seek_global_minimum=seek_global_minimum,
        )
    except Exception as exc:
        return fallback_result(theta_start, exc)

def better_result(res_a, res_b):
    if res_a is None:
        return res_b
    if res_b is None:
        return res_a
    return res_b if float(res_b.f) < float(res_a.f) else res_a

def run_local_track(theta_start):
    res1 = solve_replica_stage(
        theta_start,
        maxfun=local_stage1_maxfun,
        rhobeg=local_stage1_rhobeg,
        rhoend=local_stage1_rhoend,
        seek_global_minimum=False,
    )
    res2 = solve_replica_stage(
        np.clip(res1.x, 0.0, 1.0),
        maxfun=local_stage2_maxfun,
        rhobeg=local_stage2_rhobeg,
        rhoend=local_stage2_rhoend,
        seek_global_minimum=False,
    )
    return better_result(res1, res2), int(res1.nf) + int(res2.nf)

def run_rescue_track(theta_start):
    res1 = solve_replica_stage(
        theta_start,
        maxfun=rescue_stage1_maxfun,
        rhobeg=rescue_stage1_rhobeg,
        rhoend=rescue_stage1_rhoend,
        seek_global_minimum=True,
    )
    res2 = solve_replica_stage(
        np.clip(res1.x, 0.0, 1.0),
        maxfun=rescue_stage2_maxfun,
        rhobeg=rescue_stage2_rhobeg,
        rhoend=rescue_stage2_rhoend,
        seek_global_minimum=False,
    )
    return better_result(res1, res2), int(res1.nf) + int(res2.nf)

def fit_replica(rng):
    starts = build_replica_starts(rng)
    best_res = None
    best_label = None
    total_nfev = 0

    for idx, theta_start in enumerate(starts):
        res, stage_nfev = run_local_track(theta_start)
        total_nfev += stage_nfev
        if (best_res is None) or (float(res.f) < float(best_res.f)):
            best_res = res
            best_label = f"local-{idx}"

    best_chi2dN = float(10 ** best_res.f)
    should_run_rescue = rescue_enabled and (
        int(best_res.flag) != 0 or best_chi2dN > rescue_chi2dN_threshold
    )

    if should_run_rescue:
        rescue_starts = dedupe_theta_starts([
            np.clip(best_res.x, 0.0, 1.0),
            jitter_theta(np.clip(best_res.x, 0.0, 1.0), base_jitter_sigma, rng),
        ])

        for idx, theta_start in enumerate(rescue_starts):
            res, stage_nfev = run_rescue_track(theta_start)
            total_nfev += stage_nfev
            if float(res.f) < float(best_res.f):
                best_res = res
                best_label = f"rescue-{idx}"
    else:
        best_label = f"{best_label}|no-rescue"

    return best_res, total_nfev, best_label

def results_header():
    return ["replica_id", "pdf_replica_id", pdf_seed_column, "success", "nfev", "best_chi2dN", *param_columns]

def prepare_results_file(path):
    with path.open("w", newline="") as f:
        csv.writer(f).writerow(results_header())

def validate_replica_objective_wiring(min_delta_chi2dN=0.5):
    global data_list, current_pdf_replica_id, current_pdf_shift_vector, current_pdf_seed

    saved_data_list = data_list
    saved_pdf_replica_id = current_pdf_replica_id
    saved_pdf_shift_vector = current_pdf_shift_vector.copy()
    saved_pdf_seed = current_pdf_seed

    try:
        data_list = clone_data_list(nominal_data_list)
        current_pdf_replica_id = None
        current_pdf_shift_vector = zero_pdf_shift_vector.copy()
        current_pdf_seed = None
        nominal_chi2dN = float(objective(initial_params))

        check_rng = np.random.default_rng(20260410)
        data_list = generate_experimental_replica_data(check_rng)
        current_pdf_replica_id, current_pdf_shift_vector, current_pdf_seed = sample_pdf_replica(check_rng, -1)
        replica_chi2dN = float(objective(initial_params))

        if (replica_chi2dN - nominal_chi2dN) < min_delta_chi2dN:
            raise RuntimeError(
                "Replica chi2 wiring check failed: replica objective is too close to the nominal chi2. "
                "Restart the kernel and rerun all setup cells before launching replica refits. "
                f"nominal={nominal_chi2dN:.6f}, replica={replica_chi2dN:.6f}"
            )
    finally:
        data_list = saved_data_list
        current_pdf_replica_id = saved_pdf_replica_id
        current_pdf_shift_vector = saved_pdf_shift_vector
        current_pdf_seed = saved_pdf_seed


In [18]:
validate_replica_objective_wiring()
prepare_results_file(results_path)

replica_results = []
replica_ids = range(N_replicas)

for replica_id in tqdm(replica_ids, total=N_replicas, desc="Replica refits"):
    replica_rng = make_replica_rng(replica_id)
    data_list = generate_experimental_replica_data(replica_rng)
    current_pdf_replica_id, current_pdf_shift_vector, current_pdf_seed = sample_pdf_replica(replica_rng, replica_id)
    res, total_nfev, best_label = fit_replica(replica_rng)

    best_chi2dN = float(10 ** res.f)
    optimal_params_free = denormalize_params(res.x)

    full_params = initial_params.copy()
    full_params[free_idx] = optimal_params_free
    full_params[frozen_idx] = frozen_vals

    row = {
        "replica_id": replica_id,
        "pdf_replica_id": current_pdf_replica_id,
        pdf_seed_column: current_pdf_seed,
        "success": int(res.flag == 0),
        "nfev": int(total_nfev),
        "best_chi2dN": best_chi2dN,
    }
    for name, value in zip(param_columns, full_params):
        row[name] = float(value)

    replica_results.append(row)

    fmt = lambda x: f"{float(x):.8g}"
    with results_path.open("a", newline="") as f:
        csv.writer(f).writerow(
            [
                row["replica_id"],
                row["pdf_replica_id"],
                row[pdf_seed_column],
                row["success"],
                row["nfev"],
                fmt(row["best_chi2dN"]),
                *[fmt(row[name]) for name in param_columns],
            ]
        )

    pdf_label = current_pdf_replica_id if current_pdf_replica_id is not None else "off"
    print(
        f"Replica {replica_id}: pdf={pdf_label}, "
        f"chi2/N={best_chi2dN:.3f}, nfev={int(total_nfev)}, track={best_label}"
    )

data_list = clone_data_list(nominal_data_list)
current_pdf_replica_id = None
current_pdf_shift_vector = zero_pdf_shift_vector.copy()
current_pdf_seed = None

replica_results_df = pd.read_csv(results_path)


Replica refits:   1%|          | 1/100 [02:37<4:19:09, 157.06s/it]

Replica 0: pdf=gaussian, chi2/N=2.146, nfev=1478, track=rescue-0


Replica refits:   2%|▏         | 2/100 [04:41<3:45:05, 137.82s/it]

Replica 1: pdf=gaussian, chi2/N=1.959, nfev=1514, track=rescue-0


Replica refits:   3%|▎         | 3/100 [05:47<2:49:49, 105.05s/it]

...

Replica 97: pdf=gaussian, chi2/N=2.164, nfev=1585, track=rescue-0


Replica refits:  99%|█████████▉| 99/100 [3:05:13<01:59, 119.27s/it]

Replica 98: pdf=gaussian, chi2/N=2.205, nfev=1569, track=rescue-0


Replica refits: 100%|██████████| 100/100 [3:07:17<00:00, 112.38s/it]

Replica 99: pdf=gaussian, chi2/N=2.152, nfev=1563, track=rescue-0


In [19]:
display(replica_results_df.head())

print(f"Wrote {len(replica_results_df)} replica refits to {results_path}")
if len(replica_results_df) > 0:
    print(f"Replica ids: {int(replica_results_df['replica_id'].min())} to {int(replica_results_df['replica_id'].max())}")
else:
    print("Replica ids: none")
print()
print(replica_results_df[["best_chi2dN", "nfev"]].describe())

if use_pdf_shift:
    print()
    print(f"PDF_rep shape: {PDF_rep.shape}")
    print("PDF shift labels:")
    display(replica_results_df["pdf_replica_id"].value_counts().sort_index().rename("count").to_frame())
else:
    print()
    print("PDF replica shifts were disabled for this run.")


,replica_id,pdf_replica_id,pdf_shift_seed,success,nfev,best_chi2dN,param_0,param_1,param_2,param_3,param_4,param_5,param_6,param_7,param_8
0,0,gaussian,5830217695817921349,1,1478,2.146382,-0.000702,0.989687,-1.913334,-3.847237,1.174493,-0.286925,1.520411,0.064924,0.025609
1,1,gaussian,224308479026628678,1,1514,1.958509,-0.001271,1.028435,-2.160632,-4.609835,1.123254,-0.320567,1.545229,0.062792,0.028115
2,2,gaussian,2783458671071757400,1,838,2.051226,-0.013528,1.038007,-2.197349,-4.362353,1.416103,-0.293823,1.588158,0.060807,0.029243
3,3,gaussian,6465246186654385331,0,1620,1.925387,0.005986,1.166082,-2.805593,-4.596398,1.262345,-0.402193,1.529344,0.067809,0.035863
4,4,gaussian,8371395858288140580,0,1620,1.982044,0.015735,0.942372,-2.190581,-4.184429,0.830080,-0.333349,1.412178,0.076389,0.035614


Wrote 100 replica refits to replica_data\replica_0410.csv
Replica ids: 0 to 99

       best_chi2dN         nfev
count   100.000000   100.000000
mean      2.094124  1410.790000
std       0.120946   255.674007
min       1.756990   800.000000
25%       1.998219  1441.750000
50%       2.100504  1531.500000
75%       2.193232  1569.250000
max       2.344126  1620.000000

PDF_rep shape: (465, 99)
PDF shift labels:


,count
pdf_replica_id,
gaussian,100
